# NSynth + AFTER Representation Evaluation: Informativeness

This notebook evaluates the **informativeness** of AFTER timbre embeddings extracted from NSynth.

The goal is to train a probe on top of frozen embeddings and evaluate whether a factor of variation, initially `instrument_family`, can be predicted from the embedding space.

This notebook assumes that:

1. The reduced NSynth dataset has already been created.
2. Each split contains a valid `examples.json`.
3. AFTER timbre embeddings have already been extracted and saved as `.pt` files.
4. The folder structure is:

```text
/content/drive/MyDrive/datasets/nsynth/
├── train/
│   ├── *.wav
│   ├── examples.json
│   └── AFTER_Timbre/
│       └── *.pt
├── validation/
│   ├── *.wav
│   ├── examples.json
│   └── AFTER_Timbre/
│       └── *.pt
└── test/
    ├── *.wav
    ├── examples.json
    └── AFTER_Timbre/
        └── *.pt
```


## 1. Mount Drive and configure paths

In [5]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import sys

REPO_ROOT = Path('/content/drive/MyDrive/SYNESIS/synesis')
DATASET_ROOT = Path('/content/drive/MyDrive/datasets/nsynth')

assert REPO_ROOT.exists(), f'No encuentro el repo en {REPO_ROOT}'
assert DATASET_ROOT.exists(), f'No encuentro NSynth en {DATASET_ROOT}'

os.chdir(REPO_ROOT)

# Make sure Python can import config.features and synesis.*
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo actual:', Path.cwd())
print('Dataset:', DATASET_ROOT)
print('config/features.py existe:', (REPO_ROOT / 'config' / 'features.py').exists())
print('synesis package existe:', (REPO_ROOT / 'synesis').exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo actual: /content/drive/MyDrive/SYNESIS/synesis
Dataset: /content/drive/MyDrive/datasets/nsynth
config/features.py existe: True
synesis package existe: True


## 2. Check Python / NumPy environment

In previous runs, NumPy 2.x caused import issues with this environment.  
For this pipeline, `numpy==1.26.4` was the stable version.


In [1]:
!pip install -q --force-reinstall "numpy==1.26.4"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9

In [2]:
import numpy as np
print(np.__version__)

1.26.4


## 3. Configure the evaluation task

In [3]:
# Feature folder containing the extracted embeddings
FEATURE = 'AFTER_Timbre'

# Label to predict from the embeddings.
# Options for this NSynth dataset:
#   - 'instrument_family'  -> single-label multiclass classification
#   - 'qualities'          -> multi-label classification
LABEL = 'instrument_family'

# Synesis task name.
# For instrument_family use your existing single-label probe task.
# If you switch LABEL to 'qualities', use a task configured with BCEWithLogitsLoss.
TASK = 'classification_SLP'

# Batch size for probe training.
BATCH_SIZE_TRAIN = 64

# Reproducibility
SEED = 42

print('Feature:', FEATURE)
print('Label:', LABEL)
print('Task:', TASK)
print('Batch size train:', BATCH_SIZE_TRAIN)


Feature: AFTER_Timbre
Label: instrument_family
Task: classification_SLP
Batch size train: 64


## 4. Verify that embeddings exist

This step checks that each split contains `.pt` files inside the `AFTER_Timbre` folder.


In [6]:
for split in ['train', 'validation', 'test']:
    split_dir = DATASET_ROOT / split
    feature_dir = split_dir / FEATURE
    json_path = split_dir / 'examples.json'

    # Your reduced NSynth currently stores wavs directly in each split folder.
    # This also supports the original NSynth structure with an audio/ subfolder.
    wav_count = len(list(split_dir.glob('*.wav')))
    if wav_count == 0 and (split_dir / 'audio').exists():
        wav_count = len(list((split_dir / 'audio').glob('*.wav')))

    pt_count = len(list(feature_dir.glob('*.pt'))) if feature_dir.exists() else 0

    print(f'\nSplit: {split}')
    print('Split dir exists:', split_dir.exists())
    print('examples.json exists:', json_path.exists())
    print('Feature dir exists:', feature_dir.exists())
    print('Number of wav files:', wav_count)
    print('Number of pt embeddings:', pt_count)

    assert split_dir.exists(), f'Missing split directory: {split_dir}'
    assert json_path.exists(), f'Missing examples.json: {json_path}'
    assert feature_dir.exists(), f'Missing feature directory: {feature_dir}'
    assert pt_count > 0, f'No .pt embeddings found in {feature_dir}'



Split: train
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of wav files: 11741
Number of pt embeddings: 11741

Split: validation
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of wav files: 2516
Number of pt embeddings: 2516

Split: test
Split dir exists: True
examples.json exists: True
Feature dir exists: True
Number of wav files: 2517
Number of pt embeddings: 2517


## 4.1 Install the custom NSynth dataset inside Synesis

This cell writes `synesis/datasets/nsynth.py` and registers it in `dataset_utils.py`, so both the notebook and the CLI can load the dataset with `-d NSynth`.

It also creates a symlink from `REPO_ROOT/data/NSynth` to your Drive dataset, because the CLI call does not pass a custom `root` path.


In [ ]:
from pathlib import Path

nsynth_py = REPO_ROOT / "synesis" / "datasets" / "nsynth.py"

nsynth_py.write_text(r'''import json
from pathlib import Path
from typing import Optional, Tuple, Union

import numpy as np
import torch
from torch import Tensor
from torch.utils.data import Dataset

from config.features import configs as feature_configs
from synesis.datasets.dataset_utils import load_track


class FixedLabelEncoder:
    """Small encoder object compatible with the Synesis downstream script."""

    def __init__(self, classes):
        self.classes_ = np.array(classes)


class NSynth(Dataset):
    def __init__(
        self,
        feature: str,
        root: Union[str, Path] = "data/NSynth",
        split: Optional[str] = None,
        download: bool = False,
        feature_config: Optional[dict] = None,
        audio_format: str = "wav",
        item_format: str = "feature",
        itemization: bool = True,
        seed: int = 42,
        label: str = "instrument_family",
        transform=None,  # ignored, for compatibility
    ) -> None:
        """
        NSynth dataset for Synesis downstream probing.

        Supported labels:
            - instrument_family: single-label multiclass classification, target shape []
            - qualities: multi-label classification, target shape [10]

        Supported folder layouts:

            root/train/examples.json
            root/train/*.wav
            root/train/<feature>/*.pt

        and also:

            root/nsynth-train/examples.json
            root/nsynth-train/audio/*.wav
            root/nsynth-train/<feature>/*.pt
        """

        self.tasks = ["instrument_family_classification", "qualities_classification"]
        self.fvs = ["instrument_family", "qualities"]

        if label not in self.fvs:
            raise ValueError(f"Invalid NSynth label: {label}. Options: {self.fvs}")

        if split not in [None, "train", "validation", "valid", "test"]:
            raise ValueError(
                f"Invalid split: {split}. Options: None, 'train', 'validation', 'valid', 'test'"
            )

        self.feature = feature
        self.root = Path(root)
        self.split = "validation" if split == "valid" else split
        self.label = label
        self.item_format = item_format
        self.itemization = itemization
        self.audio_format = audio_format

        if not feature_config:
            feature_config = feature_configs[feature]
        self.feature_config = feature_config

        if download:
            self._download()

        if self.label == "instrument_family":
            # NSynth has 11 instrument families: 0..10
            self.label_encoder = FixedLabelEncoder(list(range(11)))
        else:
            # Official NSynth qualities vector has 10 positions.
            self.label_encoder = FixedLabelEncoder(
                [
                    "bright",
                    "dark",
                    "distortion",
                    "fast_decay",
                    "long_release",
                    "multiphonic",
                    "nonlinear_env",
                    "percussive",
                    "reverb",
                    "tempo-synced",
                ]
            )

        self._load_metadata()

    def _download(self) -> None:
        raise NotImplementedError(
            "NSynth download is not implemented here. Place the dataset under data/NSynth "
            "or pass root=/path/to/nsynth."
        )

    def _candidate_split_dirs(self):
        if self.split is None:
            return [
                self.root / "train",
                self.root / "validation",
                self.root / "test",
                self.root / "nsynth-train",
                self.root / "nsynth-valid",
                self.root / "nsynth-test",
            ]

        mapping = {
            "train": ["train", "nsynth-train", "nsynth_train"],
            "validation": ["validation", "valid", "nsynth-valid", "nsynth-validation", "nsynth_valid"],
            "test": ["test", "nsynth-test", "nsynth_test"],
        }
        return [self.root / name for name in mapping[self.split]]

    def _existing_split_dirs(self):
        split_dirs = [p for p in self._candidate_split_dirs() if (p / "examples.json").exists()]
        if not split_dirs:
            candidates = "\n".join(str(p / "examples.json") for p in self._candidate_split_dirs())
            raise FileNotFoundError(
                "Could not find NSynth examples.json for the requested split. Tried:\n"
                + candidates
            )
        return split_dirs

    def _audio_path_for_note(self, split_root: Path, note_id: str) -> Path:
        direct = split_root / f"{note_id}.{self.audio_format}"
        nested = split_root / "audio" / f"{note_id}.{self.audio_format}"
        if direct.exists():
            return direct
        return nested

    def _feature_path_for_note(self, split_root: Path, note_id: str) -> Path:
        return split_root / self.feature / f"{note_id}.pt"

    def _qualities_to_vector(self, qualities, note_id: str):
        if isinstance(qualities, dict):
            return [int(qualities[name]) for name in self.label_encoder.classes_]

        if len(qualities) != len(self.label_encoder.classes_):
            raise ValueError(
                f"Expected 10 NSynth qualities, got {len(qualities)} for note {note_id}"
            )

        return [int(q) for q in qualities]

    def _load_metadata(self) -> Tuple[list, torch.Tensor]:
        raw_data_paths = []
        feature_paths = []
        labels = []

        for split_root in self._existing_split_dirs():
            metadata_path = split_root / "examples.json"
            with open(metadata_path, "r") as f:
                metadata = json.load(f)

            for note_id, example in sorted(metadata.items()):
                raw_data_paths.append(self._audio_path_for_note(split_root, note_id))
                feature_paths.append(self._feature_path_for_note(split_root, note_id))

                if self.label == "instrument_family":
                    labels.append(int(example["instrument_family"]))
                elif self.label == "qualities":
                    labels.append(self._qualities_to_vector(example["qualities"], note_id))

        self.raw_data_paths = raw_data_paths
        self.feature_paths = feature_paths

        if self.label == "instrument_family":
            self.labels = torch.tensor(labels, dtype=torch.long)
        else:
            # The downstream BCEWithLogitsLoss branch casts targets to float when needed.
            self.labels = torch.tensor(labels, dtype=torch.long)

        self.paths = self.raw_data_paths if self.item_format == "raw" else self.feature_paths

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        path = self.raw_data_paths[idx] if self.item_format == "raw" else self.feature_paths[idx]
        label = self.labels[idx]

        item_len_sec = self.feature_config.get("item_len_sec", None)

        track = load_track(
            path=path,
            item_format=self.item_format,
            itemization=self.itemization,
            item_len_sec=item_len_sec,
            sample_rate=self.feature_config["sample_rate"],
        )

        return track, label


# Compatibility aliases: both import styles work.
Nsynth = NSynth
''')

print(f"Wrote {nsynth_py}")

Wrote /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/nsynth.py


## 5. Verify that the custom Nsynth dataset loads embeddings correctly

This checks that the dataset can load `.pt` embeddings and read the selected label.


In [ ]:
from config.features import configs as feature_configs
from synesis.datasets.nsynth import NSynth

feature_config = {
    **feature_configs[FEATURE],
    "item_len_sec": 4,
    "sample_rate": 44100,
}

ds_train = NSynth(
    feature=FEATURE,
    root=DATASET_ROOT,
    split="train",
    item_format="feature",
    label=LABEL,
    feature_config=feature_config,
)

print("Number of train examples:", len(ds_train))

if LABEL == "qualities":
    print("Multi-label task. Label shape:", ds_train.labels.shape)
else:
    print("Classes:", list(ds_train.label_encoder.classes_))

x, y = ds_train[0]
print("Embedding type:", type(x))
print("Embedding shape:", x.shape if hasattr(x, "shape") else "No direct shape")
print("Label:", y)


Number of train examples: 11741
Classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Embedding type: <class 'torch.Tensor'>
Embedding shape: torch.Size([1, 1, 6])
Label: tensor(0)


In [ ]:
#Celda para que apunte al path correspondiente
from pathlib import Path

nsynth_path = REPO_ROOT / "synesis" / "datasets" / "nsynth.py"
text = nsynth_path.read_text()

old = 'root: Union[str, Path] = "data/NSynth",'
new = 'root: Union[str, Path] = "/content/drive/MyDrive/datasets/nsynth",'

if old in text:
    nsynth_path.write_text(text.replace(old, new))
    print("✅ Root por defecto cambiado en nsynth.py")
else:
    print("⚠️ No he encontrado la línea exacta. Busca manualmente:")
    print(old)

print("Archivo:", nsynth_path)

✅ Root por defecto cambiado en nsynth.py
Archivo: /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/nsynth.py


In [ ]:
#Comprobar que apunta donde toca :)
import importlib
import synesis.datasets.nsynth as nsynth_module

importlib.reload(nsynth_module)
NSynth = nsynth_module.NSynth

ds_train = NSynth(
    feature=FEATURE,
    split="train",
    item_format="feature",
    label=LABEL,
    feature_config=feature_config,
)

print("Root usado:", ds_train.root)
print("Len:", len(ds_train))
x, y = ds_train[0]
print(x.shape, y)

Root usado: /content/drive/MyDrive/datasets/nsynth
Len: 11741
torch.Size([1, 1, 6]) tensor(0)


## 6. Check Synesis informativeness CLI

Before launching the experiment, inspect the available command-line options.

Look for options related to:
- feature (`-f`)
- dataset (`-d`)
- label (`-l`)
- task (`-t`)
- batch size
- epochs
- W&B / wandb logging


In [ ]:
!pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.7 MB/s eta 0:00:00


In [ ]:
!pip install -q mir_eval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 2.9 MB/s eta 0:00:00


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Torch CUDA version:", torch.version.cuda)

CUDA available: True
Torch CUDA version: 12.8


PRUEBAS :)

In [ ]:
#AÑADIMOS PRINTS EN NSYNTH.PY PARA IR HACIENDO DEBUGGING
from pathlib import Path

nsynth_path = REPO_ROOT / "synesis" / "datasets" / "nsynth.py"

text = nsynth_path.read_text()

# Activamos debug global
if "DEBUG_NSYNTH = True" not in text:
    text = text.replace(
        "from synesis.datasets.dataset_utils import load_track",
        "from synesis.datasets.dataset_utils import load_track\n\nDEBUG_NSYNTH = True",
    )

# Print al entrar en __init__
if '[NSynth DEBUG] __init__' not in text:
    text = text.replace(
        "        self._load_metadata()",
        """        if DEBUG_NSYNTH:
            print(f"[NSynth DEBUG] __init__ feature={self.feature} root={self.root} split={self.split} label={self.label} item_format={self.item_format}", flush=True)

        self._load_metadata()""",
    )

# Print en _existing_split_dirs
if '[NSynth DEBUG] candidate split dirs' not in text:
    text = text.replace(
        "    def _existing_split_dirs(self):\n        split_dirs = [",
        """    def _existing_split_dirs(self):
        if DEBUG_NSYNTH:
            print("[NSynth DEBUG] candidate split dirs:", flush=True)
            for p in self._candidate_split_dirs():
                print(f"  - {p} | examples exists={(p / 'examples.json').exists()}", flush=True)

        split_dirs = [""",
    )

# Print al leer metadata
if '[NSynth DEBUG] reading metadata' not in text:
    text = text.replace(
        "            metadata_path = split_root / \"examples.json\"\n\n            with open(metadata_path, \"r\") as f:",
        """            metadata_path = split_root / "examples.json"

            if DEBUG_NSYNTH:
                print(f"[NSynth DEBUG] reading metadata: {metadata_path}", flush=True)

            with open(metadata_path, "r") as f:""",
    )

# Print después de json.load
if '[NSynth DEBUG] loaded examples' not in text:
    text = text.replace(
        "                metadata = json.load(f)\n\n            for note_id, example in sorted(metadata.items()):",
        """                metadata = json.load(f)

            if DEBUG_NSYNTH:
                print(f"[NSynth DEBUG] loaded examples: {len(metadata)} from {metadata_path}", flush=True)

            for note_id, example in sorted(metadata.items()):""",
    )

# Print al final de _load_metadata
if '[NSynth DEBUG] total paths' not in text:
    text = text.replace(
        "        self.paths = (\n            self.raw_data_paths\n            if self.item_format == \"raw\"\n            else self.feature_paths\n        )",
        """        self.paths = (
            self.raw_data_paths
            if self.item_format == "raw"
            else self.feature_paths
        )

        if DEBUG_NSYNTH:
            print(f"[NSynth DEBUG] total paths={len(self.paths)} labels_shape={tuple(self.labels.shape)}", flush=True)
            if len(self.paths) > 0:
                print(f"[NSynth DEBUG] first path={self.paths[0]}", flush=True)
                print(f"[NSynth DEBUG] first path exists={Path(self.paths[0]).exists()}", flush=True)
                print(f"[NSynth DEBUG] first label={self.labels[0]}", flush=True)""",
    )

# Print solo para los primeros __getitem__
if '[NSynth DEBUG] __getitem__ idx=' not in text:
    text = text.replace(
        "        track = load_track(\n            path=path,",
        """        if DEBUG_NSYNTH and idx < 3:
            print(f"[NSynth DEBUG] __getitem__ idx={idx} path={path} exists={Path(path).exists()}", flush=True)

        track = load_track(
            path=path,""",
    )

# Print después de load_track
if '[NSynth DEBUG] loaded track' not in text:
    text = text.replace(
        "        return track, label",
        """        if DEBUG_NSYNTH and idx < 3:
            print(f"[NSynth DEBUG] loaded track idx={idx} shape={getattr(track, 'shape', None)} label={label}", flush=True)

        return track, label""",
    )

nsynth_path.write_text(text)

print("Debug prints añadidos en:", nsynth_path)

Debug prints añadidos en: /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/nsynth.py


In [ ]:
from synesis.datasets.nsynth import NSynth
from config.features import configs as feature_configs
import time

feature_config = {
    **feature_configs[FEATURE],
    "item_len_sec": 4,
    "sample_rate": 44100,
}

ds_train = NSynth(
    feature=FEATURE,
    root=REPO_ROOT / "data" / "NSynth",
    split="train",
    item_format="feature",
    label=LABEL,
    feature_config=feature_config,
)

print("LEN:", len(ds_train))

t0 = time.time()
x, y = ds_train[0]
print("Load ds_train[0] took:", time.time() - t0, "seconds")
print("x shape:", x.shape)
print("y:", y)

LEN: 11741
Load ds_train[0] took: 0.0042989253997802734 seconds
x shape: torch.Size([1, 1, 6])
y: tensor(0)


In [ ]:
ds_train = NSynth(
    feature=FEATURE,
    root=REPO_ROOT / "data" / "NSynth",
    split="train",
    item_format="feature",
    label=LABEL,
    feature_config=feature_config,
    itemization=False,
)

x, y = ds_train[0]
print("x shape:", x.shape)
print("y:", y)

x shape: torch.Size([1, 1, 6])
y: tensor(0)


In [ ]:
from pathlib import Path

nsynth_path = REPO_ROOT / "synesis" / "datasets" / "nsynth.py"
text = nsynth_path.read_text()

old = '''        if DEBUG_NSYNTH and idx < 3:
            print(f"[NSynth DEBUG] loaded track idx={idx} shape={getattr(track, 'shape', None)} label={label}", flush=True)

        return track, label
'''

new = '''        if self.item_format == "feature":
            track = track.reshape(-1)

        if DEBUG_NSYNTH and idx < 3:
            print(f"[NSynth DEBUG] loaded track idx={idx} shape={getattr(track, 'shape', None)} label={label}", flush=True)

        return track, label
'''

if old in text:
    nsynth_path.write_text(text.replace(old, new))
    print("✅ Reshape aplicado correctamente en:", nsynth_path)
else:
    print("⚠️ No he encontrado el bloque exacto otra vez.")
    print(text[text.find("    def __getitem__"):text.find("    def __getitem__") + 1200])

✅ Reshape aplicado correctamente en: /content/drive/MyDrive/SYNESIS/synesis/synesis/datasets/nsynth.py


In [ ]:
import importlib
import synesis.datasets.nsynth as nsynth_module

importlib.reload(nsynth_module)
NSynth = nsynth_module.NSynth

print("NSynth recargado")

NSynth recargado


In [ ]:
ds_train = NSynth(
    feature=FEATURE,
    root=REPO_ROOT / "data" / "NSynth",
    split="train",
    item_format="feature",
    label=LABEL,
    feature_config=feature_config,
)

[NSynth DEBUG] __init__ feature=AFTER_Timbre root=/content/drive/MyDrive/SYNESIS/synesis/data/NSynth split=train label=instrument_family item_format=feature
[NSynth DEBUG] candidate split dirs:
  - /content/drive/MyDrive/SYNESIS/synesis/data/NSynth/train | examples exists=True
  - /content/drive/MyDrive/SYNESIS/synesis/data/NSynth/nsynth-train | examples exists=False
  - /content/drive/MyDrive/SYNESIS/synesis/data/NSynth/nsynth_train | examples exists=False
[NSynth DEBUG] loaded examples: 11741 from /content/drive/MyDrive/SYNESIS/synesis/data/NSynth/train/examples.json


In [ ]:
x, y = ds_train[0]
print(x.shape)

[NSynth DEBUG] __getitem__ idx=0 path=/content/drive/MyDrive/SYNESIS/synesis/data/NSynth/train/AFTER_Timbre/bass_electronic_018-022-025.pt exists=True
[NSynth DEBUG] loaded track idx=0 shape=torch.Size([6]) label=0
torch.Size([6])


In [ ]:
# Comprobar que hay GPU
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU: Tesla T4


In [ ]:
!python -m synesis.informativeness.downstream \
    -f $FEATURE \
    -d NSynth \
    -t $TASK \
    -l $LABEL \
    -i feature \
    --device cuda #forzando la GPU pq si no me lo hace con la CPU :)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: nerea-gonzalez02 (nerea-gonzalez02-universitat-pompeu-fabra) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ setting up run nmh1mw54 (0.0s)
wandb: ⣻ setting up run nmh1mw54 (0.0s)
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in /content/drive/MyDrive/SYNESIS/synesis/wandb/run-20260530_101827-nmh1mw54
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run INFO_DOWN_classification_SLP_NSynth_instrument_family_AFTER_Timbre
wandb: ⭐️ View project at https://wandb.ai/nerea-gonzalez02-universitat-pompeu-fabra/synesis
wandb: 🚀 View run at https://wandb.ai/nerea-gonzalez02-universitat-pompeu-fabra/synesis/runs/nmh1mw54
[NSynth DEBUG] __init__ feature=AFTER_Timbre root=/content/drive/MyDrive/datasets/nsynth split=train label=instrument_family item_format=feature
[NSynth DEBUG] candidate split dirs:
  - /

## 7. Optional: connect to Weights & Biases

Run this if you want the probe training to be logged to W&B.

Whether this is enough depends on whether `synesis.informativeness.downstream` already supports W&B flags.  
If the CLI does not support W&B directly, the Synesis training script will need a small modification.


In [ ]:
!pip install -q wandb

import wandb
wandb.login()

## 8. Run informativeness / downstream probing

This command trains a probe to predict `instrument_family` from the frozen `AFTER_Timbre` embeddings.

If `--help` shows different argument names, adjust the command accordingly.


In [ ]:
!python -m synesis.informativeness.downstream \
    -f $FEATURE \
    -d NSynth \
    -l $LABEL \
    -t $TASK \
    -i feature \
    --nolog


## 9. Alternative command template with possible W&B flags

Use this only if the `--help` output shows that the downstream script supports W&B arguments.

The exact flag names may be different, so check the previous help cell first.


In [ ]:
# Example only. Uncomment and adapt if the Synesis CLI supports these flags.
# !python -m synesis.informativeness.downstream \
#     -f $FEATURE \
#     -d Nsynth \
#     -l $FV \
#     -t $TASK \
#     --wandb \
#     --project tfg-after-nsynth

## 10. Notes for future `qualities` experiment

`instrument_family` is a **single-label** classification task.

`qualities` is different: it is a **multi-label** task, because one sound can have several qualities at the same time.

For `qualities`, the probe training code needs:
- output dimension equal to the number of quality labels,
- `BCEWithLogitsLoss`,
- multi-label metrics such as micro-F1 / macro-F1.

The embeddings do **not** need to be re-extracted. Only the probing task changes.


In [ ]:
# Future configuration for qualities:
# LABEL = 'qualities'
#
# IMPORTANT:
# TASK must be a Synesis task configured for multi-label classification:
#   - criterion: torch.nn.BCEWithLogitsLoss
#   - output dimension: 10
#   - metrics: multi-label metrics
#
# The embeddings do not need to be re-extracted. Only LABEL/TASK changes.
